# Dataset de Classificação
## Dry_Bean: Dados visuais sobre 7 tipos diferentes de feijão.

VISÃO GERAL: Imagens de 13.611 grãos de 7 tipos diferentes de feijão foram capturadas com uma câmera de alta resolução. Foram obtidas 16 características dos grãos — 12 relacionadas a dimensões e 4 a formas —, além da variável de classe:
- Área: A área ocupada pelo grão, representada pelo número de pixels dentro de seus limites.
- Perímetro: A circunferência do grão, definida como o comprimento de seu contorno.
- MajorAxisLength (Comprimento do Eixo Maior): A distância entre as extremidades da linha mais longa que pode ser traçada no grão.
- MinorAxisLength (Comprimento do Eixo Menor): A linha mais longa que pode ser traçada no grão, perpendicular ao eixo maior.
- AspectRatio (Proporção de Aspecto): Define a relação entre o comprimento do eixo maior e o comprimento do eixo menor.
- Eccentricity (Excentricidade): A excentricidade da elipse que possui os mesmos momentos que a região do grão.
- ConvexArea (Área Convexa): O número de pixels no menor polígono convexo que pode conter a área do grão.
- EquivDiameter (Diâmetro Equivalente): O diâmetro de um círculo que possui a mesma área que o grão.
- Extent (Extensão): A razão entre o número de pixels na caixa delimitadora (bounding box) e a área do grão.
- Solidity (Solidez): Também conhecida como convexidade. A razão entre o número de pixels no envoltório convexo e o número de pixels que compõem o grão.
- Roundness (Arredondamento): Calculado pela fórmula: (4π × Área) / (Perímetro²).
- Compactness (Compacidade): Mede o grau de arredondamento de um objeto.
- ShapeFactor1 (Fator de Forma 1): Define a relação entre o comprimento do eixo maior e a área.
- ShapeFactor2 (Fator de Forma 2): Define a relação entre o comprimento do eixo menor e a área.
- ShapeFactor3 (Fator de Forma 3): Define a relação entre a área do grão e um círculo derivado de seu eixo maior.
- ShapeFactor4 (Fator de Forma 4): Define a relação entre a área do grão e um círculo derivado de seu eixo menor.
- Classe: Seker, Barbunya, Bombay, Cali, Dermosan, Horoz e Sira.

site:
https://archive.ics.uci.edu/dataset/602/dry+bean+dataset

In [ ]:
# Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Imports
import sys
from pathlib import Path

# Base project directory
BASE = Path('/content/drive/MyDrive/ML-Project')

# Add BASE directory to Python path
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

# Import all functions from GraphicFunctions.py
from GraphicFunctions import *

print("GraphicFunctions carregado com sucesso!")

Mounted at /content/drive


In [ ]:
!pip install -q ucimlrepo "altair>=5" vl-convert-python
import altair as alt
import numpy as np
import requests
import tarfile
import io
import pandas as pd
import numpy as np

In [ ]:
import warnings
from IPython.display import display
warnings.filterwarnings("ignore")

alt.data_transformers.disable_max_rows()

CLASS_COLORS = {
    "DERMASON": "#0173B2",
    "SIRA":     "#DE8F05",
    "SEKER":    "#029E73",
    "HOROZ":    "#D55E00",
    "CALI":     "#CC78BC",
    "BARBUNYA": "#CA9161",
    "BOMBAY":   "#FBAFE4",
}

def stylize(chart):
    return (
        chart
        .configure_axis(labelFontSize=10, titleFontSize=11, gridColor="#E5E5E5")
        .configure_legend(symbolOpacity=1, labelFontSize=10, titleFontSize=11)
        .configure_title(fontSize=14)
    )

In [ ]:
from ucimlrepo import fetch_ucirepo

dry_bean = fetch_ucirepo(id=602)
df = pd.concat([dry_bean.data.features, dry_bean.data.targets], axis=1)

df = df.rename(columns={"AspectRation": "AspectRatio", "roundness": "Roundness"})

TARGET = "Class"
FEATURES = ["Area", "Perimeter", "MajorAxisLength", "MinorAxisLength",
    "AspectRatio", "Eccentricity", "ConvexArea", "EquivDiameter", "Extent", "Solidity",
    "Roundness", "Compactness", "ShapeFactor1", "ShapeFactor2", "ShapeFactor3", "ShapeFactor4"]

## Análise exploratória

### Estrutura Geral

In [ ]:
print("Shape:", df.shape)
print("\nTipos de dados:")
print(df.dtypes)

print("\nValores ausentes por coluna:")
print(df.isna().sum())
print("Total de ausentes:", int(df.isna().sum().sum()))

n_dup = int(df.duplicated().sum())
print(f"\nLinhas duplicadas: {n_dup}")

print("\nPrimeiras linhas:")
display(df.head())

print("\nÚltimas linhas:")
display(df.tail())

cont = df[TARGET].value_counts()
CLASS_ORDER = cont.index.tolist()

CLASS_SCALE = alt.Scale(domain=CLASS_ORDER, range=[CLASS_COLORS[c] for c in CLASS_ORDER])

def class_color(legend=True):
    return alt.Color(
        f"{TARGET}:N", scale=CLASS_SCALE, title="Classe",
        legend=alt.Legend(title="Classe") if legend else None,
    )

Em uma primeira análise, descobrimos que não há valores ausentes ou colunas faltantes. O número de linhas duplicadas também é baixo (apenas 68 para mais de 13000), o que significa que sua remoção não prejudicará o dataset.

Conforme esperado, quase todas as features são contínuas. As exceções são Area e ConvexArea, pois são quantificadas em pixels, logo inteiras. Dito isso, todas as features são numéricas, sendo diretamente aceitas pelos modelos que serão usados no futuro.

### Análise da variável alvo

In [ ]:
# Histograma da distribuição de classes
df_cls = pd.DataFrame({
    TARGET: CLASS_ORDER,
    "count": [int(cont[c]) for c in CLASS_ORDER]
})

perc = df[TARGET].value_counts(normalize=True).mul(100).round(2)

df_cls["count"] = [int(cont[c]) for c in CLASS_ORDER]
df_cls["percentage"] = [f"({perc[c]:.1f}%)" for c in CLASS_ORDER]

base = alt.Chart(df_cls).encode(
            x = alt.X(f"{TARGET}:N", sort=CLASS_ORDER, title="Classe", axis=alt.Axis(labelAngle=0)),
            y = alt.Y("count:Q", title="Número de instâncias")
        )

bars = base.mark_bar().encode(color=class_color(legend=False))
labels_n = base.mark_text(baseline="bottom", dy=-16, fontSize=11).encode(text=alt.Text("count:Q", format="d"))
labels_p = base.mark_text(baseline="bottom", dy=-4, fontSize=11).encode(text="percentage:N")

hist_cls = (bars + labels_n + labels_p).properties(
    width=560, height=320, title="Distribuição das classes de feijão")
display(hist_cls)

ratio = cont.max() / cont.min()
print(f"Classe majoritária: {cont.idxmax()} ({cont.max()}) | Classe minoritária: {cont.idxmin()} ({cont.min()})")
print(f"Razão entre as duas classes: {ratio:.2f}")

A análise da distribuição da variável alvo mostra um desbalanceamento considerável, com a classe mais comum sendo quase 7 vezes mais frequente que a mais rara. Isso sugere que a acurácia não é a métrica mais adequada para um modelo. Alternativas como F1-score podem ser melhores.

### Análise das features numéricas

In [ ]:
# Histogramas das features
def hist_feature(f):
    return (
        alt.Chart(df[[f]])
        .mark_bar()
        .encode(
            x = alt.X(f"{f}:Q", bin=alt.Bin(maxbins=40), title=None),
            y = alt.Y("count():Q", title=None)
        )
        .properties(
            width=190, height=130,
            title=alt.TitleParams(f, subtitle=f"(assimetria = {df[f].skew():.2f})",
                                  fontSize=11, subtitleFontSize=10)
        )
    )


all_hists = alt.concat(*[hist_feature(f) for f in FEATURES], columns=4).properties(
    title=alt.TitleParams("Histogramas das 16 features", fontSize=15, anchor="middle"))
display(all_hists)

In [ ]:
# Boxplots das features
box_plts = (
    alt.Chart(df[FEATURES])
    .mark_boxplot(extent=1.5, size=40,
                  median={"color": "black"}, outliers={"size": 6})
    .encode(y=alt.Y(alt.repeat("repeat"), type="quantitative"))
    .properties(width=120, height=130)
    .repeat(repeat=FEATURES, columns=4)
    .resolve_scale(y="independent")
    .properties(title=alt.TitleParams(
        "Boxplots das 16 features",
        fontSize=15, anchor="middle"))
)

display(box_plts)

Em todas as features relacionadas a tamanho, nota-se uma forte assimetria positiva. Ainda durante a análise de dados, vale verificar se isso acontece porque classes mais raras de feijão possuem grãos maiores.

As features  Solidity e ShapeFactor4 possuem variância visivelmente baixa, fato que será reforçado em outras análises.

Além disso, muitas features mostram ter pelo menos dois picos. Logicamente, classes muito distintas de feijão concentram a distribuição delas em pontos específicos. Essas features podem ser especialmente úteis para predição.

In [ ]:
# Tabela de estatísticas descritivas
stats = pd.DataFrame({
    "min": df[FEATURES].min(),
    "max": df[FEATURES].max(),
    "media": df[FEATURES].mean(),
    "desvio": df[FEATURES].std(),
    "assimetria": df[FEATURES].skew(),
})

stats["CV"] = stats["desvio"] / stats["media"]
print("Estatísticas descritivas:")
display(stats)

print(f"\nMaior desvio-padrão: {stats['desvio'].idxmax()} = {stats['desvio'].max():,.2f}")
print(f"Menor desvio-padrão: {stats['desvio'].idxmin()} = {stats['desvio'].min():,.6f}")
print(f"Razão entre maior e menor desvio: {stats['desvio'].max() / stats['desvio'].min():,.0f}")

A tabela de estatísticas mostra uma diferença extrema nas ordens de grandeza entre as features. De fato, muitas features representam razões intrinsecamente limitadas ao intervalo [0, 1], enquanto outras como Area e ConvexArea são representadas por quantidades de pixels em fotos de alta resolução, atingindo a ordem de dezenas de milhares. A padronização se torna essencial.

In [ ]:
# Outliers por feature
q1 = df[FEATURES].quantile(0.25)
q3 = df[FEATURES].quantile(0.75)
iqr = q3 - q1
outls = ((df[FEATURES] < q1 - 1.5 * iqr) | (df[FEATURES] > q3 + 1.5 * iqr)).sum() # Regra IQR
out_tab = pd.DataFrame({"Outliers": outls, "Porcentagem%": (outls / len(df) * 100).round(2)})
print("Outliers por feature:")
display(out_tab.sort_values("Outliers", ascending=False))

### Relação entre features e alvo

In [ ]:
# Boxplots por classe
box_plts_cls = (
    alt.Chart(df[FEATURES + [TARGET]])
    .mark_boxplot(extent=1.5, size=12, median={"color": "black"}, outliers={"size": 4})
    .encode(
        x = alt.X(f"{TARGET}:N", sort=CLASS_ORDER, title=None,
                axis=alt.Axis(labelAngle=-60, labelFontSize=8)),
        y = alt.Y(alt.repeat("repeat"), type="quantitative"),
        color=class_color(legend=False)
    )
    .properties(width=150, height=130)
    .repeat(repeat=FEATURES, columns=4)
    .resolve_scale(y="independent")
    .properties(title=alt.TitleParams("Distribuição de cada feature por classe",
                                      fontSize=15, anchor="middle"))
)

display(box_plts_cls)

alt.RepeatChart(...)

É razoável supor que algumas classes de feijão são muito diferentes em tamanho ou forma. Para testar essa hipótese, fizemos alguns gráficos de dispersão usando apenas pares de features dimensionais OU morfológicas escolhidos por intuição, e observamos como as classes se separam nesses gráficos:

In [ ]:
# Alguns gráficos de dispersão especiais
PAIRS = [
    ("Area", "Perimeter"),
    ("MajorAxisLength", "MinorAxisLength"),
    ("ShapeFactor1", "AspectRatio")
]

brush = alt.selection_interval()

for a, b in PAIRS:
    chart = (
        alt.Chart(df[[a, b, TARGET]])
        .mark_circle(size=14, opacity=0.5, cursor="crosshair")
        .encode(
            x=alt.X(f"{a}:Q", scale=alt.Scale(zero=False)),
            y=alt.Y(f"{b}:Q", scale=alt.Scale(zero=False)),
            color=class_color(),
            tooltip=[
              alt.Tooltip(f"{a}:Q", format=".2f"),
              alt.Tooltip(f"{b}:Q", format=".2f"),
              alt.Tooltip(f"{TARGET}:N", title="Classe")
            ]
        )
        .add_params(brush)
        .interactive()
        .properties(width=480, height=360, title=f"{a} x {b}")
    )
    display(chart)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

Eis o que podemor notar em cada gráfico:

*   **Gráfico 1:** Mesmo sem nenhum modelo, a classe Bombay é facilmente identificável nesse gráfico. Além disso, há uma relação quase linear entre Area e Perimeter, mas a relação entre features será explorada mais tarde.
*   **Gráfico 2:** A correlação entre as features também existe, mas é mais fraca que no gráfico anterior. Com menos redundância, a divisão entre as classes é mais clara, porém Bombay continua se destacando.
*   **Gráfico 3:** Essas features foram escolhidas por ter CV alto mesmo sendo morfológicas. O que mais chama atenção é a semelhança com o gráfico 2, mostrando que features dimensionais e morfológicas carregam informação parecida.

Adicionalmente, notamos que nas 4 features relacionadas a tamanho a classe Bombay supera visivelmente as demais. Isso justifica as assimetrias positivas que observamos nessas features anteriormente, já que essa é a classe minoritária.

### Relações entre features

In [ ]:
# Matriz de correlação de Pearson
corr = df[FEATURES].corr(method="pearson")

pos = {f: i for i, f in enumerate(FEATURES)}
corr_long = (
    corr.rename_axis("feature a").reset_index()
    .melt(id_vars="feature a", var_name="feature b", value_name="r")
)
corr_long = corr_long[corr_long["feature a"].map(pos) > corr_long["feature b"].map(pos)]
corr_long["r_txt"] = corr_long["r"].map(lambda v: f"{v:.2f}")

base = alt.Chart(corr_long).encode(
    x=alt.X("feature b:N", scale=alt.Scale(domain=FEATURES), title=None, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("feature a:N", scale=alt.Scale(domain=FEATURES), title=None)
)

cells = base.mark_rect(stroke="white", strokeWidth=0.5).encode(
    color=alt.Color(
        "r:Q",
        scale=alt.Scale(scheme="redblue", reverse=True, domain=[-1, 1]),
        legend=alt.Legend(title="Correlação de Pearson", gradientLength=300)
    )
)

values = base.mark_text(fontSize=8).encode(
    text="r_txt:N",
    color=alt.condition("abs(datum.r) > 0.5", alt.value("white"), alt.value("black")),
)

corr_heatmap = (cells + values).properties(
    width=16 * 38, height=16 * 38,
    title="Matriz da correlação de Pearson entre as 16 features")
display(corr_heatmap)

idx_i, idx_j = np.triu_indices(len(FEATURES), k=1)
pairs = pd.DataFrame({
    "feature a": [FEATURES[i] for i in idx_i],
    "feature b": [FEATURES[j] for j in idx_j],
    "r": corr.values[idx_i, idx_j],
})
pairs["|r|"] = pairs["r"].abs()
highest = pairs[pairs["|r|"] > 0.9].sort_values("|r|", ascending=False).reset_index(drop=True)
print(f"De {len(pairs)} pares, {len(highest)} possuem |r| > 0.9:")
display(highest)

alt.LayerChart(...)

De 120 pares, 21 possuem |r| > 0.9:


,feature a,feature b,r,|r|
0,Area,ConvexArea,0.999939,0.999939
1,Compactness,ShapeFactor3,0.998686,0.998686
2,Perimeter,EquivDiameter,0.991380,0.991380
3,AspectRatio,Compactness,-0.987687,0.987687
4,ConvexArea,EquivDiameter,0.985226,0.985226
5,Area,EquivDiameter,0.984968,0.984968
6,Eccentricity,ShapeFactor3,-0.981058,0.981058
7,AspectRatio,ShapeFactor3,-0.978592,0.978592
8,Perimeter,MajorAxisLength,0.977338,0.977338
9,Eccentricity,Compactness,-0.970313,0.970313


Temos um alerta importante: vários pares de features são fortemente correlacionados. É natural, uma vez todas são relacionadas ao tamanho ou forma do grão.

Do ponto de vista de modelos que dependam da distância ou possuem alto custo computacional, isso significa que esse dataset possui uma dimensionalidade desnecessariamente alta. Também surge o problema da multicolinearidade, que afeta métodos lineares.

A implementação de métodos de redução de dimensionalidade, como o PCA, torna se fortemente recomendada.

### Aprendizado não supervisionado

#### PCA

Como todas as features são numéricas, usamos todas para o PCA. Por padrão, normalizamos previamente.

In [ ]:
FEATURES_CLS = FEATURES

X_cls = df[FEATURES_CLS].to_numpy(dtype=float)
Z_cls = normalize(X_cls)
pca_cls, S_cls, tab_pca_cls, var_exp_cls, var_acum_cls = fit_pca(Z_cls, FEATURES_CLS)
display(tab_pca_cls)

,PC1,PC2,PC3,PC4
Area,0.282458,0.245882,0.061447,-0.031546
Perimeter,0.310891,0.179303,0.018853,-0.042468
MajorAxisLength,0.325824,0.100757,0.084692,-0.006793
MinorAxisLength,0.236199,0.343461,-0.007500,-0.061300
AspectRatio,0.229298,-0.330844,0.169058,0.053646
Eccentricity,0.231526,-0.319434,0.163042,0.118389
ConvexArea,0.283200,0.244630,0.053649,-0.030960
EquivDiameter,0.297484,0.222802,0.049914,-0.032427
Extent,-0.059808,0.220619,0.085258,0.948254
Solidity,-0.143016,0.103322,0.738670,-0.049546


In [ ]:
# Gráfico do cotovelo do PCA
display(pca_elbow_chart(var_exp_cls, var_acum_cls))

alt.LayerChart(...)

O PCA se mostra efetivo ao concentrar mais de 90% da variância em apenas 4 componentes. Tomando essas 4 componentes, resolvemos o problema da alta dimensionalidade perdendo pouca informação.

Quanto as cargas, notamos que PC1 favorece features associadas a tamanho, como Perimeter e MajorAxisLength. Já PC2 é alto para grãos arredondados e de baixa excentricidade. As componentes PC3 e PC4 pesam apenas em componentes com CV baixo, não tendo tanto significado.

#### K-means

In [ ]:
models_cls, wss_cls, tab_wss_cls = sweep_k(S_cls)
display(tab_wss_cls)

,K,WSS,Queda absoluta,Queda %
0,2,119250.56,0.00,0.00
1,3,100214.84,19035.71,15.96
2,4,67465.55,32749.29,32.68
3,5,53389.57,14075.97,20.86
4,6,47484.57,5905.01,11.06
5,7,45318.00,2166.56,4.56
6,8,43605.87,1712.14,3.78
7,9,37331.06,6274.81,14.39
8,10,34921.00,2410.06,6.46


In [ ]:
# Gráfico do cotovelo K-means
display(wss_chart(wss_cls))

alt.Chart(...)

Para a aplicação de KMeans, construímos o gráfico da WSS e argumentamos que é possível visualizar um cotovelo em K=5, justificando essa escolha de hiperparâmetro. Podemos analisar o perfil dos clusters pela proporção de classes:

In [ ]:
# Proporção de classes por cluster
K_CLS = 5
km_cls = KMeans(k=K_CLS, seed=SEED).fit(S_cls)
cluster_labels = [f"C{i}" for i in range(K_CLS)]

df_cluster = pd.DataFrame({
    "cluster": [cluster_labels[c] for c in km_cls.labels],
    TARGET: df[TARGET].to_numpy(),
})

cluster_class_counts = (
    df_cluster.groupby(["cluster", TARGET])
    .size()
    .reset_index(name="count")
)

cluster_totals = cluster_class_counts.groupby("cluster")["count"].transform("sum")
cluster_class_counts["proportion"] = cluster_class_counts["count"] / cluster_totals

cluster_class_chart = (
    alt.Chart(cluster_class_counts)
    .mark_bar()
    .encode(
        x=alt.X("cluster:N", sort=cluster_labels, title="Cluster"),
        y=alt.Y("proportion:Q", title="Proporção de classes", axis=alt.Axis(format="%")),
        color=class_color(),
        order=alt.Order(f"{TARGET}:N", sort="ascending"),
        tooltip=[
            alt.Tooltip("cluster:N", title="Cluster"),
            alt.Tooltip(f"{TARGET}:N", title="Classe"),
            alt.Tooltip("count:Q", title="Contagem"),
            alt.Tooltip("proportion:Q", title="Proporção", format=".1%")
        ]
    )
    .properties(
        width=480, height=360,
        title="Proporção de classes por cluster (K-means)"
    )
)

display(cluster_class_chart)

alt.Chart(...)

A escolha de K=5 gera uma configuração interessante: 3 dos 5 clusters são praticamente dominados por uma única classe. Em especial, 99.8% do cluster C1 está na classe Bombay, que é a classe mais distinta. Por outro lado, Cali e Barbunya permanecem misturadas no cluster C4.

In [ ]:
# Gráfico PCA + centroides
display(projection_chart(S_cls, km_cls, (TARGET, df[TARGET].to_numpy(), class_color())))

alt.LayerChart(...)

Por fim, tentamos visualizar a dispersão das classes usando apenas 2 componentes principais. O resultado se assemelha aos gráficos de dispersão plotados antes.

Adicionando os centroides nessa visualização, percebemos que de fato, as classes se concentram muito bem em 5 clusters. Ainda, como o número de classes é maior do que o de clusters, há fortes sobreposições em alguns pares: Dermason x Sira e Cali x Barbunya. Provavelmente serão os pares mais confundidos nos modelos de classificação.

### Verificação de plausibilidade dos dados e checagem de leakage

In [ ]:
checks = {}
checks["Area > 0"] = (df["Area"] > 0).all()
checks["Perimeter > 0"] = (df["Perimeter"] > 0).all()
checks["MajorAxisLength >= MinorAxisLength"] = (df["MajorAxisLength"] >= df["MinorAxisLength"]).all()
checks["ConvexArea >= Area"] = (df["ConvexArea"] >= df["Area"]).all()
checks["AspectRatio >= 1"] = (df["AspectRatio"] >= 1).all()

checks["Eccentricity em [0, 1)"] = ((df["Eccentricity"] >= 0) & (df["Eccentricity"] < 1)).all()

# Razões que devem estar em (0, 1].
ratios01 = ["Extent", "Solidity", "Roundness",
             "Compactness", "ShapeFactor3", "ShapeFactor4"]

for f in ratios01:
    checks[f"{f} em (0, 1]"] = ((df[f] > 0) & (df[f] <= 1)).all()

checks_df = pd.DataFrame({"Passou?": pd.Series(checks)})
display(checks_df)

if checks_df["Passou?"].all():
    print("Todas as checagens passaram!")
else:
    failures = (~checks_df["Passou?"]).sum()
    print(f"Total de checagens falhas: {failures}")

,Passou?
Area > 0,True
Perimeter > 0,True
MajorAxisLength >= MinorAxisLength,True
ConvexArea >= Area,True
AspectRatio >= 1,True
"Eccentricity em [0, 1)",True
"Extent em (0, 1]",True
"Solidity em (0, 1]",True
"Roundness em (0, 1]",True
"Compactness em (0, 1]",True


Todas as checagens passaram!


Acima, uma bateria de checagens que verifica a plausibilidade lógica dos dados. Não houve problema em nenhuma das checagens. No mais, devemos comentar que nenhuma das features é derivada do tipo do grão, portanto o risco de leakage só existe no caso de uma má divisão dos dados ao treinar os modelos.